# Entrenamiento del modelo

___

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, f1_score
import pandas as pd

df = pd.read_csv("../data/train_imputed.csv")
df['Personality'] = df['Personality'].map({"Extrovert": 0, "Introvert": 1})
X = df.drop(columns=['Personality'])
y = df['Personality']



scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X)



# Ajustamos PCA para reducir la dimensionalidad
pca = PCA(n_components=0.95)  # conserva el 95% de la varianza explicada
X_train_pca = pca.fit_transform(X_train_scaled)



# Espacio de búsqueda de hiperparámetros
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(random_state=42, class_weight='balanced')

grid_search = GridSearchCV(
    rf, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_search.fit(X_train_pca, y)

print("Mejores parámetros:", grid_search.best_params_)
print("Mejor F1 en CV:", grid_search.best_score_)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Mejores parámetros: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 100}
Mejor F1 en CV: 0.9399152567712804


In [ ]:
best_rf = grid_search.best_estimator_

df_test = pd.read_csv("../data/test_imputed.csv")

y_pred = best_rf.predict(pca.transform(scaler_final.transform(df_test)))

df_test['Personality'] = y_pred

submission = df_test[['id', 'Personality']]
submission['Personality'] = submission['Personality'].map({0: "Extrovert", 1: "Introvert"})
submission.to_csv("../data/submission.csv", index=False)
submission

,id,Personality
0,18524,Extrovert
1,18525,Introvert
2,18526,Extrovert
3,18527,Extrovert
4,18528,Introvert
...,...,...
6170,24694,Extrovert
6171,24695,Introvert
6172,24696,Extrovert
6173,24697,Extrovert
